In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
DATASET_PATH = DATA_DIR / "dataset.csv"
REPORT_DIR = DATA_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("EARLY DISEASE PREDICTION - DATA AUDIT")
print("Project directory:", PROJECT_DIR)
print("Dataset path:", DATASET_PATH)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        "Make sure dataset.csv is inside the data folder."
    )

df = pd.read_csv(DATASET_PATH)

print("\nDATASET OVERVIEW")
print("Dataset shape:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nFirst five records:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

required_columns = [
    "patient_id",
    "age",
    "bmi",
    "blood_pressure",
    "glucose",
    "cholesterol",
    "smoking",
    "label"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

extra_columns = [
    column for column in df.columns
    if column not in required_columns
]

print("\nCOLUMN VALIDATION")
print("Missing required columns:", missing_columns)
print("Additional columns:", extra_columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nCOLUMN DATA TYPES")
display(df.dtypes.astype(str).to_frame("Data Type"))

print("\nMISSING VALUES")
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})
display(missing_summary)

print("\nDUPLICATE CHECKS")
duplicate_rows = int(df.duplicated().sum())
duplicate_patient_ids = int(df["patient_id"].duplicated().sum())

print("Duplicate rows:", duplicate_rows)
print("Duplicate patient IDs:", duplicate_patient_ids)

print("\nTARGET LABEL ANALYSIS")
display(
    df["label"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Count")
)

display(
    (
        df["label"]
        .value_counts(normalize=True, dropna=False)
        .sort_index() * 100
    ).round(2).to_frame("Percentage")
)

valid_labels = bool(df["label"].dropna().isin([0, 1]).all())
print("Are all non-missing labels 0 or 1?", valid_labels)

numeric_columns = [
    "age",
    "bmi",
    "blood_pressure",
    "glucose",
    "cholesterol",
    "smoking"
]

print("\nNUMERIC FEATURE CHECKS")

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")
    print(
        f"{column}: "
        f"{df[column].isna().sum()} missing or non-numeric values"
    )

print("\nNUMERIC SUMMARY")
display(df[numeric_columns].describe().round(2))

print("\nPOTENTIALLY INVALID VALUES")

invalid_checks = {
    "age_outside_0_to_120": ~df["age"].between(0, 120),
    "bmi_nonpositive": df["bmi"] <= 0,
    "blood_pressure_nonpositive": df["blood_pressure"] <= 0,
    "glucose_nonpositive": df["glucose"] <= 0,
    "cholesterol_nonpositive": df["cholesterol"] <= 0,
    "smoking_not_0_or_1": ~df["smoking"].isin([0, 1])
}

invalid_summary = {}

for check_name, condition in invalid_checks.items():
    count = int(condition.fillna(False).sum())
    invalid_summary[check_name] = count
    print(f"{check_name}: {count}")

print("\nFEATURE CORRELATIONS WITH LABEL")

correlation_data = df[numeric_columns + ["label"]].corr(
    numeric_only=True
)

display(
    correlation_data[["label"]]
    .drop(index="label")
    .sort_values("label", ascending=False)
    .round(4)
)

audit_report = {
    "dataset_file": str(DATASET_PATH),
    "number_of_rows": int(df.shape[0]),
    "number_of_columns": int(df.shape[1]),
    "columns": df.columns.tolist(),
    "missing_values": {
        str(k): int(v)
        for k, v in df.isna().sum().items()
    },
    "duplicate_rows": duplicate_rows,
    "duplicate_patient_ids": duplicate_patient_ids,
    "label_distribution": {
        str(k): int(v)
        for k, v in df["label"].value_counts(dropna=False).items()
    },
    "valid_binary_labels": valid_labels,
    "invalid_value_counts": invalid_summary,
    "warning": (
        "Passing these checks does not establish that labels are medically "
        "correct or that the dataset represents real patients."
    )
}

report_path = REPORT_DIR / "data_audit_report.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(audit_report, file, indent=4)

missing_summary.to_csv(
    REPORT_DIR / "missing_values_report.csv"
)

print("\nDATA AUDIT COMPLETED")
print("Audit report saved to:", report_path)
print(
    "Missing-value report saved to:",
    REPORT_DIR / "missing_values_report.csv"
)

print(
    "Reminder: verify the dataset source and label definition "
    "before interpreting the model as a disease predictor."
)

EARLY DISEASE PREDICTION - DATA AUDIT
Project directory: c:\Users\saura\OneDrive\Desktop\early_disease_prediction
Dataset path: c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\dataset.csv

DATASET OVERVIEW
Dataset shape: (5000, 8)
Number of rows: 5000
Number of columns: 8

First five records:


,patient_id,age,bmi,blood_pressure,glucose,cholesterol,smoking,label
0,SYN00001,29,26.3,121,81,221,1,0
1,SYN00002,64,25.8,145,121,246,0,1
2,SYN00003,58,33.0,142,133,219,0,1
3,SYN00004,47,33.3,122,87,217,0,0
4,SYN00005,47,25.4,84,84,198,0,1



Column names:
['patient_id', 'age', 'bmi', 'blood_pressure', 'glucose', 'cholesterol', 'smoking', 'label']

COLUMN VALIDATION
Missing required columns: []
Additional columns: []

COLUMN DATA TYPES


,Data Type
patient_id,object
age,int64
bmi,float64
blood_pressure,int64
glucose,int64
cholesterol,int64
smoking,int64
label,int64



MISSING VALUES


...

Are all non-missing labels 0 or 1? True

NUMERIC FEATURE CHECKS
age: 0 missing or non-numeric values
bmi: 0 missing or non-numeric values
blood_pressure: 0 missing or non-numeric values
glucose: 0 missing or non-numeric values
cholesterol: 0 missing or non-numeric values
smoking: 0 missing or non-numeric values

NUMERIC SUMMARY


,age,bmi,blood_pressure,glucose,cholesterol,smoking
count,5000.00,5000.0,5000.00,5000.00,5000.00,5000.00
mean,49.93,28.6,121.53,116.79,213.21,0.28
std,14.76,5.5,16.31,28.83,29.94,0.45
min,25.00,18.0,65.00,70.00,150.00,0.00
25%,37.00,24.7,111.00,95.00,192.00,0.00
50%,50.00,28.5,122.00,116.00,213.00,0.00
75%,63.00,32.5,132.00,137.00,234.00,1.00
max,75.00,40.0,180.00,200.00,280.00,1.00



POTENTIALLY INVALID VALUES
age_outside_0_to_120: 0
bmi_nonpositive: 0
blood_pressure_nonpositive: 0
glucose_nonpositive: 0
cholesterol_nonpositive: 0
smoking_not_0_or_1: 0

FEATURE CORRELATIONS WITH LABEL


,label
glucose,0.4043
bmi,0.1776
age,0.1647
cholesterol,0.1042
blood_pressure,0.0896
smoking,0.0293



DATA AUDIT COMPLETED
Audit report saved to: c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\reports\data_audit_report.json
Missing-value report saved to: c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\reports\missing_values_report.csv
Reminder: verify the dataset source and label definition before interpreting the model as a disease predictor.
